# KAT Dataset Generation

## Obtained Gamelog for Last 4 Seasons

In [18]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import time

kat = players.find_players_by_full_name("Karl-Anthony Towns")[0]
kat_id = kat["id"]

seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]

all_logs = []

for season in seasons:
    for season_type in ["Regular Season", "Playoffs"]:
        try:
            log = playergamelog.PlayerGameLog(
                player_id=kat_id,
                season=season,
                season_type_all_star=season_type
            ).get_data_frames()[0]

            log["SEASON"] = season
            log["SEASON_TYPE"] = season_type

            all_logs.append(log)

            time.sleep(0.6)

        except Exception as e:
            print(f"Failed {season} {season_type}: {e}")

kat_all = pd.concat(all_logs, ignore_index=True)

print(kat_all.shape)
print(kat_all[["SEASON", "SEASON_TYPE", "GAME_DATE", "MATCHUP", "BLK"]].head())


(291, 29)
    SEASON     SEASON_TYPE     GAME_DATE      MATCHUP  BLK
0  2022-23  Regular Season  Apr 09, 2023  MIN vs. NOP    1
1  2022-23  Regular Season  Apr 08, 2023    MIN @ SAS    2
2  2022-23  Regular Season  Apr 04, 2023    MIN @ BKN    0
3  2022-23  Regular Season  Apr 02, 2023  MIN vs. POR    0
4  2022-23  Regular Season  Mar 31, 2023  MIN vs. LAL    0


### Organized Variables

In [19]:
Kat_gamelog_df = kat_all.copy()

Kat_gamelog_df["GAME_DATE"] = pd.to_datetime(Kat_gamelog_df["GAME_DATE"])

# Opponent abbreviation
Kat_gamelog_df["OPP"] = Kat_gamelog_df["MATCHUP"].str.strip().str[-3:]

# Home = 1, away = 0
Kat_gamelog_df["HOME"] = Kat_gamelog_df["MATCHUP"].str.contains("vs.").astype(int)

# Binary target for block prop
Kat_gamelog_df["Block_Stat"] = (Kat_gamelog_df["BLK"] > 0).astype(int)

# Sort oldest to newest so EMA works correctly later
Kat_gamelog_df = Kat_gamelog_df.sort_values("GAME_DATE").reset_index(drop=True)

Kat_gamelog_df = Kat_gamelog_df.drop(columns= ['MATCHUP', 'WL'])

print(Kat_gamelog_df.tail(15))
print(Kat_gamelog_df["SEASON_TYPE"].value_counts())
print(Kat_gamelog_df["OPP"].value_counts())

    SEASON_ID  Player_ID     Game_ID  GAME_DATE  MIN  FGM  FGA  FG_PCT  FG3M  \
276     22025    1626157  0022501176 2026-04-10   30    8   12   0.667     1   
277     42025    1626157  0042500121 2026-04-18   33    6   13   0.462     3   
278     42025    1626157  0042500122 2026-04-20   34    8   12   0.667     2   
279     42025    1626157  0042500123 2026-04-23   34    7   12   0.583     1   
280     42025    1626157  0042500124 2026-04-25   29    6   10   0.600     1   
281     42025    1626157  0042500125 2026-04-28   34    5    7   0.714     1   
282     42025    1626157  0042500126 2026-04-30   28    1    4   0.250     0   
283     42025    1626157  0042500211 2026-05-04   20    7   11   0.636     3   
284     42025    1626157  0042500212 2026-05-06   27    6    8   0.750     1   
285     42025    1626157  0042500213 2026-05-08   26    3    8   0.375     0   
286     42025    1626157  0042500214 2026-05-10   20    5    7   0.714     2   
287     42025    1626157  0042500301 202

## Obtained Shot Location Data for All Teams

In [20]:
from nba_api.stats.endpoints import leaguedashteamshotlocations, leaguedashteamstats
from nba_api.stats.static import teams
import pandas as pd
import time
import os

nba_teams = teams.get_teams()

name_to_abbr = {
    team["full_name"]: team["abbreviation"]
    for team in nba_teams
}

def flatten_columns(df):
    df = df.copy()

    flat_cols = []
    for col in df.columns:
        if isinstance(col, tuple):
            flat_col = "_".join([str(x) for x in col if str(x) != ""])
        else:
            flat_col = str(col)

        flat_col = (
            flat_col
            .replace(" ", "_")
            .replace("-", "_")
            .replace("(", "")
            .replace(")", "")
            .replace("%", "PCT")
            .replace("/", "_")
        )

        flat_cols.append(flat_col)

    df.columns = flat_cols
    return df

In [21]:
def get_all_team_shot_zones(season, season_type="Regular Season"):
    shot_df = leaguedashteamshotlocations.LeagueDashTeamShotLocations(
        season=season,
        season_type_all_star=season_type,
        per_mode_detailed="PerGame",
        distance_range="By Zone"
    ).get_data_frames()[0]

    shot_df = flatten_columns(shot_df)

    if "TEAM_NAME" in shot_df.columns:
        shot_df["TEAM_ABBR"] = shot_df["TEAM_NAME"].map(name_to_abbr)
    elif "TEAM_ABBREVIATION" in shot_df.columns:
        shot_df["TEAM_ABBR"] = shot_df["TEAM_ABBREVIATION"]
    else:
        raise ValueError(f"No team column found. Columns: {shot_df.columns.tolist()}")

    keep_cols = ["TEAM_ABBR"]

    zone_keywords = [
        "Restricted_Area",
        "In_The_Paint",
        "Mid_Range",
        "Left_Corner_3",
        "Right_Corner_3",
        "Above_the_Break_3",
        "Backcourt"
    ]

    stat_keywords = ["FGM", "FGA", "FG_PCT"]

    for col in shot_df.columns:
        col_lower = col.lower()

        is_zone_col = any(zone.lower() in col_lower for zone in zone_keywords)
        is_stat_col = any(stat.lower() in col_lower for stat in stat_keywords)

        if is_zone_col and is_stat_col:
            keep_cols.append(col)

    team_shot_zones = shot_df[keep_cols].copy()
    team_shot_zones["SEASON"] = season
    team_shot_zones["SEASON_TYPE"] = season_type

    return team_shot_zones

In [22]:
seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]
season_types = ["Regular Season", "Playoffs"]

all_shot_zone_tables = []

for season in seasons:
    for season_type in season_types:
        try:
            print(f"Fetching shot zones for {season} {season_type}")

            temp = get_all_team_shot_zones(season, season_type)
            all_shot_zone_tables.append(temp)

            time.sleep(0.8)

        except Exception as e:
            print(f"Failed shot zones {season} {season_type}: {e}")

team_shot_zones_all = pd.concat(all_shot_zone_tables, ignore_index=True)

print(team_shot_zones_all.shape)
print(team_shot_zones_all.head())
print(team_shot_zones_all.columns.tolist())

Fetching shot zones for 2022-23 Regular Season
Fetching shot zones for 2022-23 Playoffs
Fetching shot zones for 2023-24 Regular Season
Fetching shot zones for 2023-24 Playoffs
Fetching shot zones for 2024-25 Regular Season
Fetching shot zones for 2024-25 Playoffs
Fetching shot zones for 2025-26 Regular Season
Fetching shot zones for 2025-26 Playoffs
(184, 24)
  TEAM_ABBR  Restricted_Area_FGM  Restricted_Area_FGA  Restricted_Area_FG_PCT  \
0       ATL                 18.0                 27.1                   0.665   
1       BOS                 16.5                 23.9                   0.689   
2       BKN                 15.0                 22.4                   0.667   
3       CHA                 19.5                 30.3                   0.645   
4       CHI                 18.1                 27.2                   0.668   

   In_The_Paint_Non_RA_FGM  In_The_Paint_Non_RA_FGA  \
0                      9.1                     20.0   
1                      6.7               

In [23]:
team_shot_zones_for_merge = team_shot_zones_all.rename(columns={
    "TEAM_ABBR": "OPP"
})

Kat_gamelog_df = Kat_gamelog_df.merge(
    team_shot_zones_for_merge,
    on=["SEASON", "SEASON_TYPE", "OPP"],
    how="left"
)

print(Kat_gamelog_df.shape)
print(Kat_gamelog_df.head())

(291, 51)
  SEASON_ID  Player_ID     Game_ID  GAME_DATE  MIN  FGM  FGA  FG_PCT  FG3M  \
0     22022    1626157  0022200010 2022-10-19   36    2   10   0.200     2   
1     22022    1626157  0022200025 2022-10-21   33    9   25   0.360     2   
2     22022    1626157  0022200041 2022-10-23   30    5    9   0.556     1   
3     22022    1626157  0022200050 2022-10-24   39    9   17   0.529     2   
4     22022    1626157  0022200062 2022-10-26   38    9   14   0.643     3   

   FG3A  ...  Left_Corner_3_FG_PCT  Right_Corner_3_FGM  Right_Corner_3_FGA  \
0     7  ...                 0.388                 1.2                 2.9   
1     6  ...                 0.415                 1.9                 4.7   
2     3  ...                 0.388                 1.2                 2.9   
3     6  ...                 0.385                 1.2                 3.3   
4     5  ...                 0.385                 1.2                 3.3   

   Right_Corner_3_FG_PCT  Above_the_Break_3_FGM  Abo

## Obtained Pace Data for All Teams

In [24]:
def get_all_teams_pace(season, season_type="Regular Season"):
    pace_df = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star=season_type,
        measure_type_detailed_defense="Advanced",
        per_mode_detailed="PerGame"
    ).get_data_frames()[0]

    pace_df = flatten_columns(pace_df)

    if "TEAM_NAME" in pace_df.columns:
        pace_df["TEAM_ABBR"] = pace_df["TEAM_NAME"].map(name_to_abbr)
    elif "TEAM_ABBREVIATION" in pace_df.columns:
        pace_df["TEAM_ABBR"] = pace_df["TEAM_ABBREVIATION"]
    else:
        raise ValueError(f"No team column found. Columns: {pace_df.columns.tolist()}")

    pace_df = pace_df[["TEAM_ABBR", "PACE"]].copy()
    pace_df["SEASON"] = season
    pace_df["SEASON_TYPE"] = season_type

    return pace_df

In [25]:
all_pace_tables = []

for season in seasons:
    for season_type in season_types:
        try:
            print(f"Fetching pace for {season} {season_type}")

            temp = get_all_teams_pace(season, season_type)
            all_pace_tables.append(temp)

            time.sleep(0.8)

        except Exception as e:
            print(f"Failed pace {season} {season_type}: {e}")

team_pace_all = pd.concat(all_pace_tables, ignore_index=True)

print(team_pace_all.shape)
print(team_pace_all.head())

Fetching pace for 2022-23 Regular Season
Fetching pace for 2022-23 Playoffs
Fetching pace for 2023-24 Regular Season
Fetching pace for 2023-24 Playoffs
Fetching pace for 2024-25 Regular Season
Fetching pace for 2024-25 Playoffs
Fetching pace for 2025-26 Regular Season
Fetching pace for 2025-26 Playoffs
(184, 4)
  TEAM_ABBR    PACE   SEASON     SEASON_TYPE
0       ATL  101.56  2022-23  Regular Season
1       BOS   99.15  2022-23  Regular Season
2       BKN   98.77  2022-23  Regular Season
3       CHA  101.47  2022-23  Regular Season
4       CHI   99.18  2022-23  Regular Season


In [26]:
team_pace_for_merge = team_pace_all.rename(columns={
    "TEAM_ABBR": "OPP",
    "PACE": "OPP_PACE"
})

Kat_gamelog_df = Kat_gamelog_df.merge(
    team_pace_for_merge,
    on=["SEASON", "SEASON_TYPE", "OPP"],
    how="left"
)

print(Kat_gamelog_df.shape)
print(Kat_gamelog_df[["GAME_DATE", "OPP", "OPP_PACE"]].head())

(291, 52)
   GAME_DATE  OPP  OPP_PACE
0 2022-10-19  OKC    101.94
1 2022-10-21  UTA    101.02
2 2022-10-23  OKC    101.94
3 2022-10-24  SAS    102.07
4 2022-10-26  SAS    102.07


## Add Custom Data Points and Save Data

In [28]:
Kat_gamelog_df["GAME_DATE"] = pd.to_datetime(Kat_gamelog_df["GAME_DATE"])
Kat_gamelog_df = Kat_gamelog_df.sort_values("GAME_DATE").reset_index(drop=True)

Kat_gamelog_df["USAGE_PROXY"] = (
    Kat_gamelog_df["FGA"] +
    0.44 * Kat_gamelog_df["FTA"] +
    Kat_gamelog_df["TOV"]
)

Kat_gamelog_df["FG3A_RATE"] = Kat_gamelog_df["FG3A"] / Kat_gamelog_df["FGA"]
Kat_gamelog_df["FTA_RATE"] = Kat_gamelog_df["FTA"] / Kat_gamelog_df["FGA"]
Kat_gamelog_df["REB_PER_MIN"] = Kat_gamelog_df["REB"] / Kat_gamelog_df["MIN"]
Kat_gamelog_df["PTS_PER_MIN"] = Kat_gamelog_df["PTS"] / Kat_gamelog_df["MIN"]
Kat_gamelog_df["AST_PER_MIN"] = Kat_gamelog_df["AST"] / Kat_gamelog_df["MIN"]

os.makedirs("NBADATA", exist_ok=True)

Kat_gamelog_df.to_csv("NBADATA/kat_df.csv", index=False)
team_shot_zones_all.to_csv("NBADATA/team_shot_zones_by_season.csv", index=False)
team_pace_all.to_csv("NBADATA/team_pace_by_season.csv", index=False)

print("Saved NBADATA/kat_df.csv")
print("Saved NBADATA/team_shot_zones_by_season.csv")
print("Saved NBADATA/team_pace_by_season.csv")

Saved NBADATA/kat_df.csv
Saved NBADATA/team_shot_zones_by_season.csv
Saved NBADATA/team_pace_by_season.csv
